**Model Validation - Assignment 2**

In [25]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Importing the dataset

prices = pd.read_csv('project9/raw_price_data-1.csv', index_col=0, parse_dates=True)
rates  = pd.read_csv('project9/DTB3-1.csv', index_col=0, parse_dates=True)

print(prices.head())
print(rates.head())

missing_per_day = prices.isnull().sum(axis=1)
multi_missing = missing_per_day[missing_per_day > 1].sort_values(ascending=False)

print("Days with 2+ assets missing simultaneously:")
print(missing_per_day.value_counts().sort_index())

print("\nAll dates with 2+ missing, showing which assets:")
for date, count in multi_missing.items():
    missing_cols = prices.columns[prices.loc[date].isnull()].tolist()
    print(f"  {date.date()}  ({count} missing): {missing_cols}")



            EURUSD=X    INGA.AS       SHEL        XOM        ^AEX        ^GSPC
Date                                                                          
2006-01-02  1.181698  10.773767        NaN        NaN  440.519989          NaN
2006-01-03  1.203297  10.712100  23.337442  29.426701  441.929993  1268.800049
2006-01-04  1.211196  10.962399  23.578112  29.477022  445.000000  1273.459961
2006-01-05  1.210698  10.828181  23.293686  29.331074  443.130005  1273.479980
2006-01-06  1.215407  10.900733  23.738560  29.909849  446.109985  1285.449951
                  DTB3
observation_date      
2006-01-03        4.07
2006-01-04        4.09
2006-01-05        4.10
2006-01-06        4.12
2006-01-09        4.14
Days with 2+ assets missing simultaneously:
0    1969
1      25
2      17
3      54
4       3
5      18
Name: count, dtype: int64

All dates with 2+ missing, showing which assets:
  2010-01-01  (5 missing): ['INGA.AS', 'SHEL', 'XOM', '^AEX', '^GSPC']
  2013-01-01  (5 missing): ['INGA.A

In [22]:
# Recreating their cleaned dataset as accurately as possible

# Forward-fill: stocks/indices → max 1 day; EURUSD → max 5 days
# Per their Table 2 description:
#   "stocks and market indices: forward fill limited to one day"
#   "EUR/USD exchange rate: five-day forward filling"
#   "DTB3: similarly forward-filled with the same maximum of one day"

equity_cols = ['INGA.AS', 'SHEL', 'XOM', '^AEX', '^GSPC']
for col in equity_cols:
    prices[col] = prices[col].ffill(limit=1)

prices['EURUSD=X'] = prices['EURUSD=X'].ffill(limit=5)

rates['DTB3'] = rates['DTB3'].ffill(limit=1)

# Merge prices and rates on common date index
df = prices.join(rates, how='left')

# Drop any remaining rows with NaN
df = df.dropna()

print(f"Cleaned dataset: {len(df)} rows")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(df.shape)

# Compute log returns 
# Loan return: daily log return from annualised DTB3 + 1% credit spread (eq. 1 in report)
CREDIT_SPREAD = 0.01

log_returns = pd.DataFrame(index=df.index)

for col in equity_cols + ['EURUSD=X']:
    log_returns[col] = np.log(df[col] / df[col].shift(1))

# Loan: r_loan,t = ln(1 + (DTB3_t + spread) / 252)
log_returns['log_loan'] = np.log(1 + (df['DTB3'] / 100 + CREDIT_SPREAD) / 252)

log_returns = log_returns.dropna()

print(f"\nLog returns: {len(log_returns)} rows")
print(log_returns.describe().round(6))

# Portfolio weights (from their Table 3) 
weights = {
    'XOM':       0.15,
    'SHEL':      0.15,
    'INGA.AS':   0.10,
    '^GSPC':     0.15,
    '^AEX':      0.15,
    'EURUSD=X':  0.10,
    'log_loan':  0.20,
}

w = pd.Series(weights)
port_returns = log_returns[w.index] @ w

print(f"\nPortfolio return stats:")
print(f"  Mean:  {port_returns.mean()*100:.6f}%  (report: 0.0101%)")
print(f"  Stdev: {port_returns.std()*100:.6f}%   (report: 1.1050%)")


Cleaned dataset: 2064 rows
Date range: 2006-01-03 to 2013-12-30
(2064, 7)

Log returns: 2063 rows
           INGA.AS         SHEL          XOM         ^AEX        ^GSPC  \
count  2063.000000  2063.000000  2063.000000  2063.000000  2063.000000   
mean     -0.000323     0.000237     0.000349    -0.000049     0.000180   
std       0.035499     0.018247     0.016456     0.014596     0.013820   
min      -0.321362    -0.115582    -0.150271    -0.095903    -0.094695   
25%      -0.013873    -0.007559    -0.006913    -0.006254    -0.004316   
50%       0.000000     0.000358     0.000000     0.000325     0.000486   
75%       0.013803     0.008898     0.007960     0.006940     0.005821   
max       0.256527     0.157214     0.158630     0.100283     0.109572   

          EURUSD=X     log_loan  
count  2063.000000  2063.000000  
mean      0.000065     0.000094  
std       0.009380     0.000076  
min      -0.143324     0.000040  
25%      -0.003573     0.000042  
50%       0.000130     0.000046

In [23]:
# Data cleaning quality check
from scipy.stats import norm

# Forward-fill artifacts
print("Zero returns (stale fill days):", (log_returns == 0).sum().to_dict())

# Offsetting EUR/USD pairs — data artifact signature
eurusd = log_returns['EURUSD=X']
sig = eurusd.std()
print("\nEURUSD consecutive spike pairs (likely Yahoo data errors):")
for i in range(1, len(eurusd)):
    r0, r1 = eurusd.iloc[i-1], eurusd.iloc[i]
    if abs(r0) > 4*sig and abs(r1) > 4*sig and r0*r1 < 0:
        print(f"  {eurusd.index[i-1].date()}  {r0*100:+.2f}%  ->  {eurusd.index[i].date()}  {r1*100:+.2f}%  (net: {(r0+r1)*100:+.2f}%)")


Zero returns (stale fill days): {'INGA.AS': 39, 'SHEL': 83, 'XOM': 81, '^AEX': 30, '^GSPC': 72, 'EURUSD=X': 30, 'log_loan': 0}

EURUSD consecutive spike pairs (likely Yahoo data errors):
  2008-01-08  +5.87%  ->  2008-01-09  -6.00%  (net: -0.13%)
  2008-02-08  +7.27%  ->  2008-02-11  -7.11%  (net: +0.16%)
  2008-09-08  +5.29%  ->  2008-09-09  -6.18%  (net: -0.89%)
  2008-10-08  +9.60%  ->  2008-10-09  -9.53%  (net: +0.07%)
  2008-12-08  +15.96%  ->  2008-12-09  -14.33%  (net: +1.63%)


**Quantitative Assessment**

In [24]:
# File for later sections: save cleaned dataset and log returns
df.to_csv('cleaned_dataset.csv')
log_returns.to_csv('log_returns.csv')

print(df.head())
print(log_returns.head())

            EURUSD=X    INGA.AS       SHEL        XOM        ^AEX  \
Date                                                                
2006-01-03  1.203297  10.712100  23.337442  29.426701  441.929993   
2006-01-04  1.211196  10.962399  23.578112  29.477022  445.000000   
2006-01-05  1.210698  10.828181  23.293686  29.331074  443.130005   
2006-01-06  1.215407  10.900733  23.738560  29.909849  446.109985   
2006-01-09  1.207496  10.995047  23.705750  29.894754  447.880005   

                  ^GSPC  DTB3  
Date                           
2006-01-03  1268.800049  4.07  
2006-01-04  1273.459961  4.09  
2006-01-05  1273.479980  4.10  
2006-01-06  1285.449951  4.12  
2006-01-09  1290.150024  4.14  
             INGA.AS      SHEL       XOM      ^AEX     ^GSPC  EURUSD=X  \
Date                                                                     
2006-01-04  0.023097  0.010260  0.001709  0.006923  0.003666  0.006543   
2006-01-05 -0.012319 -0.012136 -0.004964 -0.004211  0.000016 -0.000412